In [1]:
# Cell 1. Load Pre-sleep Design C Stage 1 dataset for rolling/history feature engineering
# 원하는 결과:
# - Stage 1 split dataset과 feature columns를 로드한다.
# - split, target distribution, feature group 구성을 확인한다.
# - rolling/history feature engineering에 사용할 base feature를 확정한다.
# - 아직 feature 생성/저장은 하지 않는다.

from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

STAGE1_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage1"
STAGE1_DATASET_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_dataset_with_split.csv"
STAGE1_FEATURE_COLUMNS_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_final_feature_columns.csv"
STAGE1_METADATA_PATH = STAGE1_DIR / "metadata.json"

STAGE2_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage2_rolling_history"
STAGE2_DIR.mkdir(parents=True, exist_ok=True)

ID_COL = "participant_object_id"
TARGET_COL = "good_sleep_label"
SPLIT_COL = "pre_sleep_split"
TIME_COL = "sleep_start_datetime"

stage1_df = pd.read_csv(STAGE1_DATASET_PATH, encoding="utf-8-sig")
stage1_feature_columns_df = pd.read_csv(STAGE1_FEATURE_COLUMNS_PATH, encoding="utf-8-sig")
stage1_metadata = json.loads(STAGE1_METADATA_PATH.read_text(encoding="utf-8"))

stage1_df[TIME_COL] = pd.to_datetime(stage1_df[TIME_COL])
stage1_df["sleep_start_date"] = pd.to_datetime(stage1_df["sleep_start_date"])
stage1_df["sleep_end_datetime"] = pd.to_datetime(stage1_df["sleep_end_datetime"])
stage1_df["prediction_cutoff_datetime"] = pd.to_datetime(stage1_df["prediction_cutoff_datetime"])

stage1_df = stage1_df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)

stage1_final_features = stage1_feature_columns_df["feature"].tolist()

# rolling은 missing indicator보다는 실제 numeric base signal 중심으로 먼저 만든다.
rolling_base_feature_cols = (
    stage1_feature_columns_df
    .loc[
        stage1_feature_columns_df["feature_group"].isin(
            ["pre_sleep_intraday", "previous_day_daily", "timing"]
        ),
        "feature",
    ]
    .tolist()
)

# timing 중 cyclical calendar는 rolling 의미가 약하므로 제외하고, window/hour 정도만 유지할지 점검한다.
rolling_exclude_features = [
    "sleep_start_dayofweek_sin",
    "sleep_start_dayofweek_cos",
    "sleep_start_month_sin",
    "sleep_start_month_cos",
]

rolling_base_feature_cols = [
    feature for feature in rolling_base_feature_cols
    if feature not in rolling_exclude_features
]

missing_rolling_base_features = [
    feature for feature in rolling_base_feature_cols
    if feature not in stage1_df.columns
]

split_summary = (
    stage1_df
    .groupby(SPLIT_COL)
    .agg(
        rows=(TARGET_COL, "size"),
        participants=(ID_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
        target_0=(TARGET_COL, lambda x: int((x == 0).sum())),
        target_1=(TARGET_COL, lambda x: int((x == 1).sum())),
        min_sleep_start=(TIME_COL, "min"),
        max_sleep_start=(TIME_COL, "max"),
    )
    .reset_index()
)

feature_group_summary = (
    stage1_feature_columns_df["feature_group"]
    .value_counts()
    .reset_index(name="features")
)

rolling_base_summary = pd.DataFrame(
    {
        "feature_group": stage1_feature_columns_df.set_index("feature").loc[
            rolling_base_feature_cols,
            "feature_group",
        ].values,
        "feature": rolling_base_feature_cols,
        "missing_rate": stage1_df[rolling_base_feature_cols].isna().mean().values,
    }
)

participant_episode_summary = (
    stage1_df
    .groupby(ID_COL)
    .agg(
        episodes=("sleep_episode_id", "size"),
        target_mean=(TARGET_COL, "mean"),
        split=(SPLIT_COL, "first"),
        min_sleep_start=(TIME_COL, "min"),
        max_sleep_start=(TIME_COL, "max"),
    )
    .reset_index()
)

print("=== Stage 2 Rolling/History Setup ===")
print("stage1 dataset:", STAGE1_DATASET_PATH.relative_to(PROJECT_ROOT), STAGE1_DATASET_PATH.exists())
print("stage1 feature columns:", STAGE1_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT), STAGE1_FEATURE_COLUMNS_PATH.exists())
print("stage2 dir:", STAGE2_DIR.relative_to(PROJECT_ROOT))
print("stage1_df shape:", stage1_df.shape)
print("stage1 final features:", len(stage1_final_features))
print("rolling base features:", len(rolling_base_feature_cols))

print("\n=== Split Summary ===")
display(split_summary)

print("\n=== Stage 1 Feature Group Summary ===")
display(feature_group_summary)

print("\n=== Rolling Base Feature Summary ===")
display(rolling_base_summary)

print("\n=== Participant Episode Summary ===")
display(participant_episode_summary["episodes"].describe().to_frame().T)

print("\n=== Validation Checks ===")
print("missing rolling base features:", len(missing_rolling_base_features))
print("duplicate sleep_episode_id rows:", len(stage1_df) - stage1_df["sleep_episode_id"].nunique())
print("missing split rows:", int(stage1_df[SPLIT_COL].isna().sum()))

if len(missing_rolling_base_features) > 0:
    print("Missing features:")
    print(missing_rolling_base_features)
else:
    print("stage2 setup validation passed")

=== Stage 2 Rolling/History Setup ===
stage1 dataset: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_dataset_with_split.csv True
stage1 feature columns: data\processed\pre_sleep_forecasting\design_c_stage1\pre_sleep_design_c_stage1_final_feature_columns.csv True
stage2 dir: data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history
stage1_df shape: (3551, 87)
stage1 final features: 58
rolling base features: 29

=== Split Summary ===


,pre_sleep_split,rows,participants,target_mean,target_0,target_1,min_sleep_start,max_sleep_start
0,test,881,14,0.360953,563,318,2021-05-24 00:26:30,2022-01-21 02:25:00
1,train,2323,46,0.412398,1365,958,2021-05-24 00:08:30,2022-01-21 23:04:00
2,validation,347,9,0.351585,225,122,2021-05-24 00:16:30,2022-01-19 03:07:00



=== Stage 1 Feature Group Summary ===


,feature_group,features
0,missing_indicator,25
1,pre_sleep_intraday,18
2,previous_day_daily,9
3,timing,6



=== Rolling Base Feature Summary ===


,feature_group,feature,missing_rate
0,pre_sleep_intraday,steps_pre_sleep_sum,0.047874
1,pre_sleep_intraday,steps_pre_sleep_record_count,0.000000
2,pre_sleep_intraday,steps_pre_sleep_active_record_count,0.000000
3,pre_sleep_intraday,steps_pre_sleep_last_3h_sum,0.047874
4,pre_sleep_intraday,steps_pre_sleep_last_1h_sum,0.047874
5,pre_sleep_intraday,calories_pre_sleep_sum,0.004787
6,pre_sleep_intraday,calories_pre_sleep_record_count,0.000000
7,pre_sleep_intraday,calories_pre_sleep_last_3h_sum,0.004787
8,pre_sleep_intraday,calories_pre_sleep_last_1h_sum,0.004787
9,pre_sleep_intraday,heart_rate_pre_sleep_mean,0.028161



=== Participant Episode Summary ===


,count,mean,std,min,25%,50%,75%,max
episodes,69.0,51.463768,34.005873,1.0,26.0,58.0,66.0,220.0



=== Validation Checks ===
missing rolling base features: 0
duplicate sleep_episode_id rows: 0
missing split rows: 0
stage2 setup validation passed


In [2]:
# Cell 2. Create leakage-safe rolling/history features from previous episodes
# 원하는 결과:
# - participant별 sleep_start_datetime 기준으로 이전 episode feature만 사용한 rolling/history feature를 만든다.
# - rolling 3/7 episode mean/std/min/max, diff1, deviation from rolling mean을 생성한다.
# - current episode 값이나 target은 rolling 계산에 포함하지 않는다.
# - feature 수, missingness, history availability를 확인한다.
# - 아직 파일 저장은 하지 않는다.

ROLLING_WINDOWS = [3, 7]
ROLLING_STATS = ["mean", "std", "min", "max"]

stage2_df = stage1_df.copy()
stage2_df = stage2_df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)

created_rolling_features = []

for feature in rolling_base_feature_cols:
    group_values = stage2_df.groupby(ID_COL, group_keys=False)[feature]

    previous_value = group_values.shift(1)
    diff_feature = f"{feature}_prev_episode_diff1"
    stage2_df[diff_feature] = stage2_df[feature] - previous_value
    created_rolling_features.append(diff_feature)

    for window in ROLLING_WINDOWS:
        rolling_object = (
            previous_value
            .groupby(stage2_df[ID_COL], group_keys=False)
            .rolling(window=window, min_periods=1)
        )

        for stat in ROLLING_STATS:
            rolling_feature = f"{feature}_prev{window}_episode_{stat}"

            if stat == "mean":
                stage2_df[rolling_feature] = rolling_object.mean().reset_index(level=0, drop=True)
            elif stat == "std":
                stage2_df[rolling_feature] = (
                    rolling_object.std(ddof=0).reset_index(level=0, drop=True)
                )
            elif stat == "min":
                stage2_df[rolling_feature] = rolling_object.min().reset_index(level=0, drop=True)
            elif stat == "max":
                stage2_df[rolling_feature] = rolling_object.max().reset_index(level=0, drop=True)
            else:
                raise ValueError(f"Unsupported rolling stat: {stat}")

            created_rolling_features.append(rolling_feature)

        deviation_feature = f"{feature}_dev_from_prev{window}_episode_mean"
        mean_feature = f"{feature}_prev{window}_episode_mean"
        stage2_df[deviation_feature] = stage2_df[feature] - stage2_df[mean_feature]
        created_rolling_features.append(deviation_feature)

stage2_df["participant_episode_order"] = (
    stage2_df
    .groupby(ID_COL)
    .cumcount()
)

for window in ROLLING_WINDOWS:
    stage2_df[f"has_prev{window}_episodes"] = (
        stage2_df["participant_episode_order"] >= window
    ).astype(int)

history_indicator_cols = [
    "participant_episode_order",
] + [
    f"has_prev{window}_episodes"
    for window in ROLLING_WINDOWS
]

stage2_candidate_feature_cols = (
    stage1_final_features
    + created_rolling_features
    + history_indicator_cols
)

stage2_missing_summary = (
    stage2_df[created_rolling_features + history_indicator_cols]
    .isna()
    .mean()
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values(["missing_rate", "feature"], ascending=[False, True])
)

history_availability_summary = (
    stage2_df
    .groupby(SPLIT_COL)
    .agg(
        rows=(TARGET_COL, "size"),
        participants=(ID_COL, "nunique"),
        target_mean=(TARGET_COL, "mean"),
        mean_episode_order=("participant_episode_order", "mean"),
        rows_with_prev3=("has_prev3_episodes", "sum"),
        rows_with_prev7=("has_prev7_episodes", "sum"),
    )
    .reset_index()
)

history_availability_summary["prev3_ratio"] = (
    history_availability_summary["rows_with_prev3"]
    / history_availability_summary["rows"]
)
history_availability_summary["prev7_ratio"] = (
    history_availability_summary["rows_with_prev7"]
    / history_availability_summary["rows"]
)

rolling_feature_group_summary = pd.DataFrame(
    [
        {"feature_group": "stage1_final", "features": len(stage1_final_features)},
        {"feature_group": "rolling_history", "features": len(created_rolling_features)},
        {"feature_group": "history_indicators", "features": len(history_indicator_cols)},
        {"feature_group": "stage2_candidate_total", "features": len(stage2_candidate_feature_cols)},
    ]
)

print("=== Stage 2 Rolling/History Feature Engineering Summary ===")
print("stage2_df shape:", stage2_df.shape)
print("rolling base features:", len(rolling_base_feature_cols))
print("created rolling/history features:", len(created_rolling_features))
print("history indicator columns:", len(history_indicator_cols))
print("stage2 candidate features:", len(stage2_candidate_feature_cols))

print("\n=== Rolling Feature Group Summary ===")
display(rolling_feature_group_summary)

print("\n=== History Availability Summary ===")
display(history_availability_summary)

print("\n=== Rolling Feature Missing Summary ===")
display(stage2_missing_summary)

print("\n=== Created Rolling Feature Preview ===")
display(pd.DataFrame({"created_feature": created_rolling_features}).head(40))

print("\n=== Stage 2 Dataset Preview ===")
display(
    stage2_df[
        [
            "sleep_episode_id",
            ID_COL,
            TIME_COL,
            SPLIT_COL,
            TARGET_COL,
            "participant_episode_order",
            "has_prev3_episodes",
            "has_prev7_episodes",
        ]
        + created_rolling_features[:10]
    ].head(10)
)

print("\n=== Validation Checks ===")
print("candidate feature count:", len(stage2_candidate_feature_cols))
print("duplicate candidate features:", len(stage2_candidate_feature_cols) - len(set(stage2_candidate_feature_cols)))
print("target used in rolling features:", any(TARGET_COL in feature for feature in created_rolling_features))

if len(stage2_candidate_feature_cols) == len(set(stage2_candidate_feature_cols)):
    print("stage2 rolling/history feature creation validation passed")
else:
    print("duplicate features detected; inspect before saving")

C:\Users\human-23\AppData\Local\Temp\ipykernel_15124\331408803.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stage2_df[rolling_feature] = (
C:\Users\human-23\AppData\Local\Temp\ipykernel_15124\331408803.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stage2_df[rolling_feature] = rolling_object.min().reset_index(level=0, drop=True)
C:\Users\human-23\AppData\Local\Temp\ipykernel_15124\331408803.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

=== Stage 2 Rolling/History Feature Engineering Summary ===
stage2_df shape: (3551, 409)
rolling base features: 29
created rolling/history features: 319
history indicator columns: 3
stage2 candidate features: 380

=== Rolling Feature Group Summary ===


C:\Users\human-23\AppData\Local\Temp\ipykernel_15124\331408803.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stage2_df[rolling_feature] = rolling_object.max().reset_index(level=0, drop=True)
C:\Users\human-23\AppData\Local\Temp\ipykernel_15124\331408803.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stage2_df[deviation_feature] = stage2_df[feature] - stage2_df[mean_feature]
C:\Users\human-23\AppData\Local\Temp\ipykernel_15124\331408803.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the r

,feature_group,features
0,stage1_final,58
1,rolling_history,319
2,history_indicators,3
3,stage2_candidate_total,380



=== History Availability Summary ===


...


=== Created Rolling Feature Preview ===


,created_feature
0,steps_pre_sleep_sum_prev_episode_diff1
1,steps_pre_sleep_sum_prev3_episode_mean
2,steps_pre_sleep_sum_prev3_episode_std
3,steps_pre_sleep_sum_prev3_episode_min
4,steps_pre_sleep_sum_prev3_episode_max
5,steps_pre_sleep_sum_dev_from_prev3_episode_mean
6,steps_pre_sleep_sum_prev7_episode_mean
7,steps_pre_sleep_sum_prev7_episode_std
8,steps_pre_sleep_sum_prev7_episode_min
9,steps_pre_sleep_sum_prev7_episode_max



=== Stage 2 Dataset Preview ===


,sleep_episode_id,participant_object_id,sleep_start_datetime,pre_sleep_split,good_sleep_label,participant_episode_order,has_prev3_episodes,has_prev7_episodes,steps_pre_sleep_sum_prev_episode_diff1,steps_pre_sleep_sum_prev3_episode_mean,steps_pre_sleep_sum_prev3_episode_std,steps_pre_sleep_sum_prev3_episode_min,steps_pre_sleep_sum_prev3_episode_max,steps_pre_sleep_sum_dev_from_prev3_episode_mean,steps_pre_sleep_sum_prev7_episode_mean,steps_pre_sleep_sum_prev7_episode_std,steps_pre_sleep_sum_prev7_episode_min,steps_pre_sleep_sum_prev7_episode_max
0,621e2e8e67b776a24055b564__20210524004000__2021...,621e2e8e67b776a24055b564,2021-05-24 00:40:00,train,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,621e2e8e67b776a24055b564__20210524234830__2021...,621e2e8e67b776a24055b564,2021-05-24 23:48:30,train,1,1,0,0,8741.0,82.000000,0.000000,82.0,82.0,8741.000000,82.000000,0.000000,82.0,82.0
2,621e2e8e67b776a24055b564__20210525234630__2021...,621e2e8e67b776a24055b564,2021-05-25 23:46:30,train,1,2,0,0,897.0,4452.500000,4370.500000,82.0,8823.0,5267.500000,4452.500000,4370.500000,82.0,8823.0
3,621e2e8e67b776a24055b564__20210526232130__2021...,621e2e8e67b776a24055b564,2021-05-26 23:21:30,train,1,3,1,0,-1509.0,6208.333333,4347.422404,82.0,9720.0,2002.666667,6208.333333,4347.422404,82.0,9720.0
4,621e2e8e67b776a24055b564__20210527233700__2021...,621e2e8e67b776a24055b564,2021-05-27 23:37:00,train,1,4,1,0,796.0,8918.000000,619.698314,8211.0,9720.0,89.000000,6709.000000,3863.555681,82.0,9720.0
5,621e2e8e67b776a24055b564__20210528235000__2021...,621e2e8e67b776a24055b564,2021-05-28 23:50:00,train,1,5,1,0,3937.0,8979.333333,616.357220,8211.0,9720.0,3964.666667,7168.600000,3575.832580,82.0,9720.0
6,621e2e8e67b776a24055b564__20210530003000__2021...,621e2e8e67b776a24055b564,2021-05-30 00:30:00,train,1,6,1,0,-12921.0,10054.000000,2069.215471,8211.0,12944.0,-10031.000000,8131.166667,3910.007051,82.0,12944.0
7,621e2e8e67b776a24055b564__20210530232930__2021...,621e2e8e67b776a24055b564,2021-05-30 23:29:30,train,1,7,1,1,3773.0,7324.666667,5407.448217,23.0,12944.0,-3528.666667,6972.857143,4599.371554,23.0,12944.0
8,621e2e8e67b776a24055b564__20210531234230__2021...,621e2e8e67b776a24055b564,2021-05-31 23:42:30,train,1,8,1,1,5432.0,5587.666667,5424.980020,23.0,12944.0,3640.333333,7503.428571,3940.947995,23.0,12944.0
9,621e2e8e67b776a24055b564__20210601235830__2021...,621e2e8e67b776a24055b564,2021-06-01 23:58:30,train,1,9,1,1,-806.0,4349.000000,3778.215011,23.0,9228.0,4073.000000,7561.285714,3962.808229,23.0,12944.0



=== Validation Checks ===
candidate feature count: 380
duplicate candidate features: 0
target used in rolling features: False
stage2 rolling/history feature creation validation passed


In [3]:
# Cell 3. Save Stage 2 rolling/history dataset and create final MLP tensors
# 원하는 결과:
# - Stage 2 rolling/history dataset을 저장한다.
# - train 기준 median imputation + StandardScaler를 적용한다.
# - train zero-variance feature를 제거한다.
# - 최종 MLP tensor train/validation/test를 저장한다.
# - NaN/Inf, feature count, target distribution을 확인한다.
# - log/YYYY-MM-DD.md를 업데이트한다.

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib
import json
from datetime import datetime

STAGE2_DATASET_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_rolling_history_dataset.csv"
STAGE2_FEATURE_COLUMNS_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_feature_columns.csv"
STAGE2_FINAL_FEATURE_COLUMNS_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_final_feature_columns.csv"
STAGE2_ZERO_VARIANCE_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_zero_variance_removed_features.csv"
STAGE2_MISSING_SUMMARY_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_missing_summary.csv"
STAGE2_IMPUTER_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_median_imputer.joblib"
STAGE2_SCALER_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_standard_scaler.joblib"
STAGE2_TENSOR_DIR = STAGE2_DIR / "mlp_current_day_final"
STAGE2_TENSOR_SUMMARY_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_tensor_summary.csv"
STAGE2_METADATA_PATH = STAGE2_DIR / "metadata.json"

LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

STAGE2_TENSOR_DIR.mkdir(parents=True, exist_ok=True)

# Fragmentation warning 정리용 copy
stage2_df = stage2_df.copy()

stage2_feature_columns_df = pd.DataFrame(
    {
        "feature": stage2_candidate_feature_cols,
        "feature_group": [
            (
                "stage1_final"
                if feature in stage1_final_features
                else "rolling_history"
                if feature in created_rolling_features
                else "history_indicator"
            )
            for feature in stage2_candidate_feature_cols
        ],
    }
)

stage2_missing_summary = (
    stage2_df[stage2_candidate_feature_cols]
    .isna()
    .mean()
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values(["missing_rate", "feature"], ascending=[False, True])
)

train_mask = stage2_df[SPLIT_COL] == "train"

X_raw = stage2_df[stage2_candidate_feature_cols].copy()
y = stage2_df[TARGET_COL].astype(np.int64).to_numpy()

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(X_raw.loc[train_mask])
scaler.fit(X_train_imputed)

X_imputed = imputer.transform(X_raw)
X_scaled = scaler.transform(X_imputed).astype(np.float32)

train_scaled = X_scaled[train_mask.to_numpy()]
train_feature_std = train_scaled.std(axis=0)

zero_variance_mask = train_feature_std <= 1e-8

stage2_zero_variance_features = [
    feature
    for feature, is_zero in zip(stage2_candidate_feature_cols, zero_variance_mask)
    if is_zero
]

stage2_final_feature_cols = [
    feature
    for feature, is_zero in zip(stage2_candidate_feature_cols, zero_variance_mask)
    if not is_zero
]

X_scaled_final = X_scaled[:, ~zero_variance_mask].astype(np.float32)

stage2_zero_variance_df = pd.DataFrame(
    {
        "removed_feature": stage2_zero_variance_features,
        "train_scaled_std": train_feature_std[zero_variance_mask],
    }
)

stage2_final_feature_columns_df = stage2_feature_columns_df[
    stage2_feature_columns_df["feature"].isin(stage2_final_feature_cols)
].copy()

tensor_summary_rows = []

for split_name in ["train", "validation", "test"]:
    split_mask = stage2_df[SPLIT_COL].to_numpy() == split_name

    X_split = X_scaled_final[split_mask]
    y_split = y[split_mask]
    episode_ids_split = stage2_df.loc[split_mask, "sleep_episode_id"].astype(str).to_numpy()
    participant_ids_split = stage2_df.loc[split_mask, ID_COL].astype(str).to_numpy()

    output_path = STAGE2_TENSOR_DIR / f"{split_name}.npz"

    np.savez_compressed(
        output_path,
        X=X_split,
        y=y_split,
        sleep_episode_id=episode_ids_split,
        participant_object_id=participant_ids_split,
        feature_columns=np.array(stage2_final_feature_cols, dtype=object),
    )

    tensor_summary_rows.append(
        {
            "tensor_type": "mlp_current_day_final",
            "split": split_name,
            "samples": int(X_split.shape[0]),
            "features": int(X_split.shape[1]),
            "participants": int(pd.Series(participant_ids_split).nunique()),
            "target_0": int((y_split == 0).sum()),
            "target_1": int((y_split == 1).sum()),
            "target_mean": float(y_split.mean()),
            "nan_count": int(np.isnan(X_split).sum()),
            "inf_count": int(np.isinf(X_split).sum()),
            "path": str(output_path.relative_to(PROJECT_ROOT)),
        }
    )

stage2_tensor_summary_df = pd.DataFrame(tensor_summary_rows)

stage2_scaled_check = pd.DataFrame(
    [
        {"metric": "rows", "value": X_scaled_final.shape[0]},
        {"metric": "candidate_features", "value": len(stage2_candidate_feature_cols)},
        {"metric": "zero_variance_features", "value": len(stage2_zero_variance_features)},
        {"metric": "final_features", "value": len(stage2_final_feature_cols)},
        {"metric": "nan_count", "value": int(np.isnan(X_scaled_final).sum())},
        {"metric": "inf_count", "value": int(np.isinf(X_scaled_final).sum())},
        {
            "metric": "max_abs_train_scaled_mean",
            "value": float(np.abs(X_scaled_final[train_mask.to_numpy()].mean(axis=0)).max()),
        },
        {
            "metric": "min_train_scaled_std",
            "value": float(X_scaled_final[train_mask.to_numpy()].std(axis=0).min()),
        },
        {
            "metric": "max_train_scaled_std",
            "value": float(X_scaled_final[train_mask.to_numpy()].std(axis=0).max()),
        },
    ]
)

stage2_df.to_csv(STAGE2_DATASET_PATH, index=False, encoding="utf-8-sig")
stage2_feature_columns_df.to_csv(STAGE2_FEATURE_COLUMNS_PATH, index=False, encoding="utf-8-sig")
stage2_final_feature_columns_df.to_csv(STAGE2_FINAL_FEATURE_COLUMNS_PATH, index=False, encoding="utf-8-sig")
stage2_zero_variance_df.to_csv(STAGE2_ZERO_VARIANCE_PATH, index=False, encoding="utf-8-sig")
stage2_missing_summary.to_csv(STAGE2_MISSING_SUMMARY_PATH, index=False, encoding="utf-8-sig")
stage2_tensor_summary_df.to_csv(STAGE2_TENSOR_SUMMARY_PATH, index=False, encoding="utf-8-sig")

joblib.dump(imputer, STAGE2_IMPUTER_PATH)
joblib.dump(scaler, STAGE2_SCALER_PATH)

stage2_metadata = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_name": "pre_sleep_design_c_stage2_rolling_history",
    "source_stage1_dataset": str(STAGE1_DATASET_PATH.relative_to(PROJECT_ROOT)),
    "prediction_target": "good_sleep_label for the sleep episode starting at sleep_start_datetime",
    "prediction_cutoff": "sleep_start_datetime",
    "strict_rule": "Rolling/history features use previous episodes only via shift(1).",
    "rows": int(len(stage2_df)),
    "participants": int(stage2_df[ID_COL].nunique()),
    "stage1_final_features": int(len(stage1_final_features)),
    "rolling_base_features": int(len(rolling_base_feature_cols)),
    "created_rolling_features": int(len(created_rolling_features)),
    "history_indicator_features": int(len(history_indicator_cols)),
    "candidate_feature_count": int(len(stage2_candidate_feature_cols)),
    "zero_variance_removed_features": int(len(stage2_zero_variance_features)),
    "final_feature_count": int(len(stage2_final_feature_cols)),
    "rolling_windows": ROLLING_WINDOWS,
    "rolling_stats": ROLLING_STATS,
    "split_column": SPLIT_COL,
    "outputs": {
        "dataset": str(STAGE2_DATASET_PATH.relative_to(PROJECT_ROOT)),
        "feature_columns": str(STAGE2_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT)),
        "final_feature_columns": str(STAGE2_FINAL_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT)),
        "zero_variance_removed_features": str(STAGE2_ZERO_VARIANCE_PATH.relative_to(PROJECT_ROOT)),
        "missing_summary": str(STAGE2_MISSING_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
        "tensor_dir": str(STAGE2_TENSOR_DIR.relative_to(PROJECT_ROOT)),
        "tensor_summary": str(STAGE2_TENSOR_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
        "imputer": str(STAGE2_IMPUTER_PATH.relative_to(PROJECT_ROOT)),
        "scaler": str(STAGE2_SCALER_PATH.relative_to(PROJECT_ROOT)),
    },
}

STAGE2_METADATA_PATH.write_text(
    json.dumps(stage2_metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

log_entry = f"""

## 2026-06-29 - Pre-sleep Design C Stage 2 rolling/history tensors

Created Stage 2 rolling/history features and final MLP tensors.

- Notebook: `notebooks/13_pre_sleep_stage2_rolling_history_features.ipynb`
- Dataset: `{STAGE2_DATASET_PATH.relative_to(PROJECT_ROOT)}`
- Tensor dir: `{STAGE2_TENSOR_DIR.relative_to(PROJECT_ROOT)}`
- Rows: {len(stage2_df)}
- Participants: {stage2_df[ID_COL].nunique()}
- Stage 1 final features: {len(stage1_final_features)}
- Rolling base features: {len(rolling_base_feature_cols)}
- Created rolling/history features: {len(created_rolling_features)}
- Candidate features: {len(stage2_candidate_feature_cols)}
- Removed zero-variance features: {len(stage2_zero_variance_features)}
- Final features: {len(stage2_final_feature_cols)}
- NaN/Inf after final tensor creation: {int(np.isnan(X_scaled_final).sum())} / {int(np.isinf(X_scaled_final).sum())}
- Leakage rule: rolling/history features use prior episodes only with `shift(1)`.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("=== Stage 2 Dataset And Final MLP Tensors Saved ===")
print("dataset:", STAGE2_DATASET_PATH.relative_to(PROJECT_ROOT), STAGE2_DATASET_PATH.exists())
print("feature columns:", STAGE2_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT), STAGE2_FEATURE_COLUMNS_PATH.exists())
print("final feature columns:", STAGE2_FINAL_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT), STAGE2_FINAL_FEATURE_COLUMNS_PATH.exists())
print("zero variance:", STAGE2_ZERO_VARIANCE_PATH.relative_to(PROJECT_ROOT), STAGE2_ZERO_VARIANCE_PATH.exists())
print("missing summary:", STAGE2_MISSING_SUMMARY_PATH.relative_to(PROJECT_ROOT), STAGE2_MISSING_SUMMARY_PATH.exists())
print("metadata:", STAGE2_METADATA_PATH.relative_to(PROJECT_ROOT), STAGE2_METADATA_PATH.exists())

print("\n=== Stage 2 Scaled Feature Check ===")
display(stage2_scaled_check)

print("\n=== Stage 2 Feature Group Counts ===")
display(stage2_final_feature_columns_df["feature_group"].value_counts().reset_index(name="features"))

print("\n=== Stage 2 Tensor Summary ===")
display(stage2_tensor_summary_df)

print("\n=== Removed Zero-Variance Feature Count ===")
print(len(stage2_zero_variance_features))
display(stage2_zero_variance_df.head(40))

print("\n=== Saved Tensor Files ===")
for split_name in ["train", "validation", "test"]:
    path = STAGE2_TENSOR_DIR / f"{split_name}.npz"
    print(path.relative_to(PROJECT_ROOT), path.exists())

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Stage 2 Dataset And Final MLP Tensors Saved ===
dataset: data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\pre_sleep_design_c_stage2_rolling_history_dataset.csv True
feature columns: data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\pre_sleep_design_c_stage2_feature_columns.csv True
final feature columns: data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\pre_sleep_design_c_stage2_final_feature_columns.csv True
zero variance: data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\pre_sleep_design_c_stage2_zero_variance_removed_features.csv True
missing summary: data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\pre_sleep_design_c_stage2_missing_summary.csv True
metadata: data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\metadata.json True

=== Stage 2 Scaled Feature Check ===


,metric,value
0,rows,3551.000000
1,candidate_features,380.000000
2,zero_variance_features,0.000000
3,final_features,380.000000
4,nan_count,0.000000
5,inf_count,0.000000
6,max_abs_train_scaled_mean,0.000001
7,min_train_scaled_std,0.999991
8,max_train_scaled_std,1.000011



=== Stage 2 Feature Group Counts ===


,feature_group,features
0,rolling_history,319
1,stage1_final,58
2,history_indicator,3



=== Stage 2 Tensor Summary ===


,tensor_type,split,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count,path
0,mlp_current_day_final,train,2323,380,46,1365,958,0.412398,0,0,data\processed\pre_sleep_forecasting\design_c_...
1,mlp_current_day_final,validation,347,380,9,225,122,0.351585,0,0,data\processed\pre_sleep_forecasting\design_c_...
2,mlp_current_day_final,test,881,380,14,563,318,0.360953,0,0,data\processed\pre_sleep_forecasting\design_c_...



=== Removed Zero-Variance Feature Count ===
0


,removed_feature,train_scaled_std



=== Saved Tensor Files ===
data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\mlp_current_day_final\train.npz True
data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\mlp_current_day_final\validation.npz True
data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\mlp_current_day_final\test.npz True

log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
